# StockEcho Event Risk 재현 실험

Event Study dataset의 시간순 분할에서 경험분포, Logistic Regression, 작은 MLP를 비교합니다. fixture 결과는 운영 승격 근거가 아니며 실제 dataset manifest를 함께 보존합니다.

In [ ]:
!git clone -q https://github.com/kimminjunnn/StockEcho.git
%cd StockEcho
!pip install -q -r requirements-modeling.txt

아래 셀에서 `events.jsonl`과 `manifest.json`을 업로드합니다. 두 파일은 `collector.jobs.export_event_study_dataset`이 같은 디렉터리에 생성합니다.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'events.jsonl' in uploaded and 'manifest.json' in uploaded
from pathlib import Path
Path('/content/events.jsonl').write_bytes(uploaded['events.jsonl'])
Path('/content/manifest.json').write_bytes(uploaded['manifest.json'])

In [ ]:
import json
manifest = json.loads(Path('/content/manifest.json').read_text())
rows = [json.loads(line) for line in Path('/content/events.jsonl').read_text().splitlines() if line]
print(manifest)
assert manifest['rowCount'] == len(rows)
assert {'train', 'validation', 'test'} == {row['split'] for row in rows}

In [ ]:
from collector.jobs.evaluate_risk_models import evaluate_rows
reports = {}
for horizon in ('d1', 'd5', 'd20'):
    label = f'material_downside_label_{horizon}'
    usable = [row for row in rows if row.get(label) in (0, 1)]
    report, _models = evaluate_rows(usable, label_key=label, fixture=False)
    reports[horizon] = report
reports

In [ ]:
from collector.risk_model.validation import leave_one_group_out_splits
company_holdouts = leave_one_group_out_splits(rows, group_key='stock_code', minimum_test_events=5)
category_holdouts = leave_one_group_out_splits(rows, group_key='event_category', minimum_test_events=5)
print('company holdouts', [x['held_out_group'] for x in company_holdouts])
print('category holdouts', [x['held_out_group'] for x in category_holdouts])

In [ ]:
artifact = {'datasetManifest': manifest, 'riskModelReports': reports,
            'companyHoldoutCount': len(company_holdouts),
            'categoryHoldoutCount': len(category_holdouts)}
Path('/content/stockecho_experiment_report.json').write_text(json.dumps(artifact, ensure_ascii=False, indent=2))
files.download('/content/stockecho_experiment_report.json')